# Notebook 1: Reading Protein Sequences with Biopython

**Chem 274A · Lab Week 11 · Sequence Similarity Networks**

<div class="alert alert-block alert-info">
<h3>Learning goals</h3>

- Know what a sequence similarity network (SSN) is and what we will build today.
- Read a FASTA file with Biopython's `SeqIO` module.
- Explore the `SeqRecord` and `Seq` classes that Biopython uses to represent sequences.
</div>

## What are we building today?

A **sequence similarity network (SSN)** is a graph where:

- each **node** is a protein sequence, and
- an **edge** connects two proteins if their sequences are similar enough (above a cutoff you choose).

Proteins with similar sequences usually have similar structures and functions, so SSNs are used to
sort large protein families into groups and to guess what an uncharacterized protein does.
Web tools like the [EFI Enzyme Similarity Tool](https://efi.igb.illinois.edu/efi-est/) do this for you:
you paste in sequences, it runs BLAST on every pair, and it gives you back a network to open in Cytoscape.

In this lab we will **build that pipeline ourselves in Python**, one step at a time:

| Notebook | Step | Tool |
|---|---|---|
| 1 | Read the sequences | Biopython `SeqIO` |
| 2 | Compare two sequences | Biopython `PairwiseAligner` |
| 3 | Compare *every* pair and build the network | `itertools` + NetworkX |
| 3 | Check the odd one out with 3D structure | Biopython `PDB` + TM-align |
| 4 (bonus) | Do it with current research tools | MMseqs2 and Foldseek |

[Biopython](https://biopython.org) is one of the most widely used Python packages in bioinformatics.
It has been developed by scientists since 1999 and is used for everything from parsing files to
running alignments and talking to online databases.

### Our dataset: globins

The file `data/globins.fasta` contains 74 **globin** proteins, the protein family that binds O₂ with a heme group.
It contains six kinds of globin from many species:

| Type | Where it is found |
|---|---|
| hemoglobin α and β | red blood cells (the two halves of the hemoglobin tetramer) |
| myoglobin | muscle |
| neuroglobin | brain and nervous system |
| cytoglobin | many tissues |
| leghemoglobin | root nodules of plants like soybeans and peas |

Three sequences are labeled `mystery_1`, `mystery_2` and `mystery_3`. **By the end of Notebook 3 you will use
your network to figure out what they are.**

## The FASTA file format

FASTA is the most common file format for sequences. Each sequence has a header line that starts with `>`,
followed by one or more lines of sequence (one letter per amino acid).

Let's look at the first few lines of the file using plain Python.

In [ ]:
with open("data/globins.fasta") as f:
    for i in range(8):
        print(f.readline(), end="")

The header line from UniProt (the main protein database) packs in a lot of information:

```
>sp|P69905|HBA_HUMAN Hemoglobin subunit alpha OS=Homo sapiens OX=9606 GN=HBA1 PE=1 SV=2
 ^^ ^^^^^^ ^^^^^^^^^ ^^^^^^^^^^^^^^^^^^^^^^^^ ^^^^^^^^^^^^^^^ ^^^^^^^ ^^^^^^^
 |    |       |          protein name           organism      taxonomy  gene name
 |    |       entry name
 |    accession (the unique ID you'd search for on uniprot.org)
 database (sp = Swiss-Prot, the hand-reviewed part of UniProt)
```

## Reading sequences with `SeqIO`

Instead of writing our own parser, we can use Biopython's `SeqIO` module. `SeqIO.parse` takes a filename and a
format name and gives back one object per sequence.

In [ ]:
from Bio import SeqIO

records = list(SeqIO.parse("data/globins.fasta", "fasta"))
print(len(records))

In [ ]:
first = records[0]
print(first)

### The `SeqRecord` class

Each item in `records` is an instance of a class called `SeqRecord`. You already know how to write classes,
so let's look at someone else's. A `SeqRecord` stores the sequence together with information about it:

In [ ]:
print(type(first))
print("id:         ", first.id)
print("description:", first.description)
print("seq:        ", first.seq)
print("length:     ", len(first))

### What's inside a `SeqRecord`?

`dir()` lists everything an object has. The cell below keeps the public names (no leading underscore) and splits them
into **attributes** (data) and **methods** (functions you can call). In Jupyter you can also type `first?` to read the
documentation.

Look at the type of `first.seq` too. And `len(first)` works because the class defines the special method `__len__`.

In [ ]:
public = [name for name in dir(first) if not name.startswith("_")]
attributes = [n for n in public if not callable(getattr(first, n))]
methods = [n for n in public if callable(getattr(first, n))]
print("attributes:", attributes)
print("methods:   ", methods)

print(type(first.seq))          # a Biopython Seq object
print("__len__" in dir(first))  # len() calls __len__

### The `Seq` class

`first.seq` is a `Seq` object. It acts a lot like a string (you can index it, slice it, and loop over it),
but it also knows biology. For example, a DNA `Seq` can be translated into protein:

In [ ]:
from Bio.Seq import Seq

print(first.seq[:10])           # slicing works like a string
print(first.seq.count("H"))     # how many histidines?
print(str(first.seq)[:10])      # convert to a plain str when you need one

dna = Seq("ATGGTGCATCTG")
print(dna.translate())          # the first four amino acids of human hemoglobin beta!

<div class="alert alert-block alert-success">
<h3>Try it: Count and index</h3>

`first` is human hemoglobin α. In the cell below:

1. Count how many cysteines (`C`) it has.
2. Find its 50th amino acid. Python counts from 0, so the 50th is at index 49.
</div>

In [ ]:
# Your code here

## Looking up sequences by ID

A list is fine for looping, but to look up a sequence by name we want a **dictionary**.
`SeqIO.to_dict` builds one for us, using each record's `id` as the key:

In [ ]:
seq_dict = SeqIO.to_dict(records)
print(seq_dict["sp|P69905|HBA_HUMAN"].description)

### Finding a sequence by its accession

Typing the full ID `sp|P69905|HBA_HUMAN` is clunky. Usually you only know the **accession**, such as `P68871`.
`find_by_accession` splits each ID on `|` and compares the middle part. The `len(parts) > 1` check skips the
`mystery_*` IDs, which have no `|`.

In [ ]:
def find_by_accession(records, accession):
    for record in records:
        parts = record.id.split("|")
        if len(parts) > 1 and parts[1] == accession:
            return record
    return None

hbb = find_by_accession(records, "P68871")
print(hbb.description)
print(hbb.seq[:20])

<div class="alert alert-block alert-success">
<h3>Try it: Look up an accession</h3>

Use `find_by_accession` to look up `P02144`. What protein is it, and which organism is it from? How many amino acids
long is it?
</div>

In [ ]:
# Your code here

## Metadata

To save you from parsing every header, `data/globins_metadata.csv` lists each sequence's protein type and
organism. We'll load it with pandas and use it in Notebook 3 to color the network.

In [ ]:
import pandas as pd

meta = pd.read_csv("data/globins_metadata.csv")
meta.head()

In [ ]:
meta["protein_type"].value_counts()

## Which proteins are longest?

pandas can split a table into groups of rows that share a value, then summarize each group.
Run the cell below. Which type is longest? Keep your answer in mind: it will matter in Notebook 2.


In [ ]:
meta.groupby("protein_type")["length"].mean().sort_values()

<div class="alert alert-block alert-success">
<h3>Try it: Shortest and longest</h3>

The cell above gives averages. Now look at individual sequences: how long are the shortest and the longest ones, and
what protein types are they?

*Hint:* `meta.sort_values("length")` sorts the table by length, and `.head()` and `.tail()` show each end.
</div>

In [ ]:
# Your code here

## Summary

- FASTA files store sequences, and Biopython's `SeqIO.parse` reads them into `SeqRecord` objects.
- A `SeqRecord` holds an `id`, a `description` and a `seq` (a `Seq` object that acts like a string).
- `SeqIO.to_dict` gives you a dictionary for quick lookup.

**Next:** Notebook 2, where we compare two sequences.